# NodeSubstrates - Basic Usage

This notebook demonstrates the basic functionality of the NodeSubstrates widget.

In [1]:
# Install dependencies (run in terminal, not here)
# cd .. && uv sync --extra dev && uv run jupyter lab

In [1]:
import networkx as nx
import numpy as np
from node_substrates import NodeSubstratesWidget

## 1. Create a Sample Graph

We'll use the Karate Club graph as a simple example, adding some numeric attributes.

In [2]:
# Create graph
G = nx.karate_club_graph()

# Add node attributes
for node in G.nodes():
    G.nodes[node]['degree'] = G.degree(node)
    G.nodes[node]['clustering'] = nx.clustering(G, node)
    G.nodes[node]['betweenness'] = nx.betweenness_centrality(G)[node]
    G.nodes[node]['closeness'] = nx.closeness_centrality(G)[node]
    
print(f"Graph has {G.number_of_nodes()} nodes and {G.number_of_edges()} edges")
print(f"Node attributes: {list(G.nodes[0].keys())}")

Graph has 34 nodes and 78 edges
Node attributes: ['club', 'degree', 'clustering', 'betweenness', 'closeness']


## 2. Create and Display Widget

In [3]:
# Create widget with auto-substrate (creates substrate from top suggestion)
widget = NodeSubstratesWidget(G, auto_substrate=True)
widget

Loading cached layout from /Users/sjoerdvink/Developer/PhD Projects/nodeSubstratesImpl/data/layout_spring_Zachary_s_Karate_Club.json


## 3. View Auto-Detection Suggestions

The widget automatically detects regions that might benefit from substrate visualization.

In [5]:
# View suggestions
for i, suggestion in enumerate(widget.suggested_regions):
    print(f"\nSuggestion {i}:")
    print(f"  Label: {suggestion['label']}")
    print(f"  Nodes: {len(suggestion['node_ids'])}")
    print(f"  Score: {suggestion['score']:.2f}")
    print(f"  Reason: {suggestion['reason']}")
    print(f"  Recommended DR: {suggestion['recommended_dr']}")


Suggestion 0:
  Label: Community 3
  Nodes: 13
  Score: 0.63
  Reason: 13 nodes with high attribute diversity, potential outliers among neighbors
  Recommended DR: pca

Suggestion 1:
  Label: Community 1
  Nodes: 17
  Score: 0.62
  Reason: 17 nodes with high attribute diversity, potential outliers among neighbors
  Recommended DR: umap


## 4. Create a Substrate

Accept a suggestion or manually create a substrate from selected nodes.

In [6]:
# Accept the first suggestion
if widget.suggested_regions:
    substrate_id = widget.accept_suggestion(0)
    print(f"Created substrate: {substrate_id}")

Created substrate: substrate_1


In [7]:
# Or create manually from specific nodes
# widget.create_substrate([0, 1, 2, 3, 4, 5], dr_method='pca', label='Core Members')

## 5. Change DR Method

Try different dimensionality reduction methods.

In [8]:
# Update to UMAP
if widget.substrates:
    widget.update_dr_method(widget.substrates[0]['id'], 'umap')
    print("Updated to UMAP")

/home/christinoleo/Projects/papers/nodeSubstrates/implementation/.venv/lib/python3.13/site-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(
/home/christinoleo/Projects/papers/nodeSubstrates/implementation/.venv/lib/python3.13/site-packages/umap/umap_.py:2462: UserWarning: n_neighbors is larger than the dataset size; truncating to X.shape[0] - 1
  warn(


Updated to UMAP


In [9]:
# Or try t-SNE
if widget.substrates:
    widget.update_dr_method(widget.substrates[0]['id'], 'tsne')
    print("Updated to t-SNE")

Updated to t-SNE


## 6. Dissolve Substrate

Return nodes to the force-directed layout.

In [10]:
# Dissolve the substrate
if widget.substrates:
    widget.dissolve_substrate(widget.substrates[0]['id'])
    print("Substrate dissolved")

Substrate dissolved


## 7. Check Selection

Use Shift+drag to lasso select nodes, or click to select individual nodes.

In [11]:
# Check selected nodes
print(f"Selected nodes: {widget.selected_nodes}")

Selected nodes: []


In [12]:
# Create substrate from selection
if len(widget.selected_nodes) >= 3:
    widget.create_substrate(widget.selected_nodes, dr_method='pca', label='Selected')
    print(f"Created substrate from {len(widget.selected_nodes)} selected nodes")